# VibeRange deterministic evidence reconstruction

This notebook reconstructs exactly one experiment run from inspectable ClickHouse views. Every result retains source event or artifact identity; correlation methods remain explicit.

In [ ]:
RUN_ID = "run-identity-proof-004"
CLICKHOUSE_URL = "http://127.0.0.1:8123"

In [ ]:
import html
import json
import urllib.parse
import urllib.request
from IPython.display import Markdown, SVG, display

def query(sql):
    url = CLICKHOUSE_URL.rstrip('/') + '/?' + urllib.parse.urlencode({'query': sql + ' FORMAT JSON'})
    with urllib.request.urlopen(url, timeout=60) as response:
        return json.load(response)['data']

def markdown_table(rows, columns):
    if not rows:
        display(Markdown('_No rows._'))
        return
    lines = ['| ' + ' | '.join(columns) + ' |', '| ' + ' | '.join(['---'] * len(columns)) + ' |']
    for row in rows:
        values = [str(row.get(column, '')).replace('|', '\\|').replace('\n', ' ')[:180] for column in columns]
        lines.append('| ' + ' | '.join(values) + ' |')
    display(Markdown('\n'.join(lines)))

## 0. Ludus range map

This map is derived from the canonical Ludus topology and the analysis VM network prefix. It makes the VM/IP roles explicit before the forensic report begins.


In [ ]:
import socket

range_identity = query(f"SELECT range_id FROM ir.experiment_runs WHERE run_id = '{RUN_ID}' LIMIT 1")
assert range_identity, f'unknown run: {RUN_ID}'
range_id = range_identity[0]['range_id']

probe = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
try:
    probe.connect(('10.255.255.255', 1))
    analysis_ip = probe.getsockname()[0]
finally:
    probe.close()
octets = analysis_ip.split('.')
assert len(octets) == 4 and octets[0] == '10', f'unexpected analysis IP: {analysis_ip}'
prefix = '.'.join(octets[:2])

ludus_map = [
    {'role': 'Analysis / defender', 'vm': f'{range_id}-clickhouse-ir', 'ip': f'{prefix}.10.10', 'services': 'ClickHouse :8123; CH-UI :3488; Jupyter :8889'},
    {'role': 'Attacker', 'vm': f'{range_id}-attacker', 'ip': f'{prefix}.20.20', 'services': 'Caldera :8888'},
    {'role': 'Victim', 'vm': f'{range_id}-k3s-vuln', 'ip': f'{prefix}.30.30', 'services': 'k3s :6443; Kubernetes Goat :1234'},
    {'role': 'Router / evidence gateway', 'vm': f'{range_id}-router-debian12-x64', 'ip': f'{prefix}.10.254 / {prefix}.20.254 / {prefix}.30.254', 'services': 'PCAP capture; Zeek'},
]
display(Markdown(f'**Range:** `{range_id}` &nbsp; **Network prefix:** `{prefix}.0.0/16`'))
markdown_table(ludus_map, ['role', 'vm', 'ip', 'services'])


## 1. Run manifest and boundary

In [ ]:
runs = query(f"SELECT run_id, range_id, scenario_id, scenario_version, blueprint_version, source_ref, status, started_at, ended_at, duration_ms, caldera_operation_id, manifest_path FROM ir.experiment_runs WHERE run_id = '{RUN_ID}'")
assert len(runs) == 1, f'expected one run, found {len(runs)}'
assert runs[0]['status'] in ('completed', 'failed', 'aborted'), runs[0]
markdown_table(runs, ['run_id', 'range_id', 'scenario_id', 'scenario_version', 'blueprint_version', 'source_ref', 'status', 'started_at', 'ended_at', 'duration_ms'])

## 2. Required evidence coverage

In [ ]:
coverage = query(f"SELECT source_type, events, coverage FROM ir.experiment_coverage WHERE run_id = '{RUN_ID}' ORDER BY source_type")
required = {'linux_auditd', 'kubernetes_audit', 'tetragon', 'zeek', 'pcap_artifact'}
present = {row['source_type'] for row in coverage if row['coverage'] == 'present'}
assert required <= present, f'missing evidence: {sorted(required - present)}'
markdown_table(coverage, ['source_type', 'events', 'coverage'])

## 3. Timeline overview

The full timeline is retained in ClickHouse. This compact view avoids an unreadable, page-width event dump while showing which sensors contributed and when each sensor first and last observed the run.

In [ ]:
timeline_summary = query(f"SELECT dataset, membership_method, count() AS events, min(event_time) AS first_seen, max(event_time) AS last_seen FROM ir.timeline WHERE run_id = '{RUN_ID}' GROUP BY dataset, membership_method ORDER BY first_seen")
assert timeline_summary, 'timeline is empty'
markdown_table(timeline_summary, ['dataset', 'events', 'first_seen', 'last_seen', 'membership_method'])

## 4. Chronological attack timeline and ability-to-evidence graph

The compact table orders the seven prepared Caldera abilities by delegated time and summarizes the independent sensors observed in each bounded execution window. The graph below retains link identity, evidence counts, marker matches, and sample event IDs for drill-down.

In [ ]:
from datetime import datetime

ABILITY_META = {
    'c0a1de4a-701d-4c9a-b541-936ff3871000': ('Discover Goat services', ''),
    'c0a1de4a-701d-4c9a-b541-936ff3871002': ('Open bounded GoTTY foothold', 'VIBERANGE_FOOTHOLD'),
    'c0a1de4a-701d-4c9a-b541-936ff3871005': ('Prove container root', 'VIBERANGE_CONTAINER_ROOT'),
    'c0a1de4a-701d-4c9a-b541-936ff3871008': ('Probe service-account RBAC', 'VIBERANGE_SERVICE_ACCOUNT_PROBE'),
    'c0a1de4a-701d-4c9a-b541-936ff3871009': ('Prove host-mount escape', 'VIBERANGE_HOST_ESCAPE'),
    'c0a1de4a-701d-4c9a-b541-936ff3871010': ('Prove bounded host control', 'VIBERANGE_HOST_CONTROL'),
    'c0a1de4a-701d-4c9a-b541-936ff3871011': ('Query namespaces as cluster admin', 'VIBERANGE_CLUSTER_ADMIN'),
}
operation_id = runs[0]['caldera_operation_id']
links = query(f"SELECT ability_id, link_id, status, delegated_at, finished_at, dateDiff('millisecond', assumeNotNull(delegated_at), assumeNotNull(finished_at)) AS duration_ms FROM ir.caldera_links WHERE operation_id = '{operation_id}' ORDER BY delegated_at")
assert links, 'run has no exported Caldera links'

candidate_events = query(f"""
SELECT event_time, dataset, event_id, source_host, process, namespace, pod_name, raw_data
FROM ir.timeline
WHERE run_id = '{RUN_ID}'
  AND dataset IN ('linux_auditd', 'tetragon', 'kubernetes_audit', 'container_log')
ORDER BY event_time
""")

def payload(row):
    try:
        value = json.loads(row.get('raw_data') or '{}')
        return value if isinstance(value, dict) else {}
    except json.JSONDecodeError:
        return {}

def exec_process(row):
    return payload(row).get('process_exec', {}).get('process', {})

def exec_parent(row):
    return payload(row).get('process_exec', {}).get('parent', {})

def audit_fields(row):
    value = payload(row)
    return {
        'verb': value.get('verb', ''),
        'uri': value.get('requestURI', ''),
        'user': value.get('user', {}).get('username', ''),
        'code': value.get('responseStatus', {}).get('code', 0),
        'resource': value.get('objectRef', {}).get('resource', ''),
        'namespace': value.get('objectRef', {}).get('namespace', ''),
    }

def ts(value):
    return datetime.fromisoformat(str(value).replace(' ', 'T').replace('Z', '+00:00'))

def closest(rows, target):
    return min(rows, key=lambda row: abs((ts(row['event_time']) - ts(target)).total_seconds())) if rows else None

def proc_candidates(binary=None, arguments=None, parent_binary=None):
    rows = []
    for row in candidate_events:
        if row['dataset'] != 'tetragon':
            continue
        process = exec_process(row)
        parent = exec_parent(row)
        if not process:
            continue
        if binary and process.get('binary') != binary:
            continue
        if arguments and arguments not in process.get('arguments', ''):
            continue
        if parent_binary and parent.get('binary') != parent_binary:
            continue
        rows.append(row)
    return rows

def victim_proof(link):
    ability = link['ability_id']
    if ability.endswith('1000'):
        rows = [row for row in candidate_events if row['dataset'] == 'container_log' and 'GET / HTTP/1.1" 200' in row['raw_data'] and 'curl/' in row['raw_data']]
        return closest(rows, link['delegated_at'])
    if ability.endswith('1002'):
        return closest(proc_candidates(binary='/usr/bin/bash'), link['delegated_at'])
    if ability.endswith('1005'):
        return closest(proc_candidates(binary='/usr/bin/id', parent_binary='/usr/bin/bash'), link['delegated_at'])
    if ability.endswith('1008'):
        rows = [row for row in candidate_events if row['dataset'] == 'kubernetes_audit' and audit_fields(row)['resource'] == 'secrets' and audit_fields(row)['namespace'] == 'default' and audit_fields(row)['user'] == 'system:serviceaccount:default:default' and audit_fields(row)['code'] == 403]
        return closest(rows, link['delegated_at'])
    if ability.endswith('1009'):
        return closest(proc_candidates(binary='/usr/sbin/chroot', arguments='/host-system /usr/bin/id'), link['delegated_at'])
    if ability.endswith('1010'):
        return closest(proc_candidates(binary='/usr/sbin/chroot', arguments='/host-system /bin/hostname'), link['delegated_at'])
    if ability.endswith('1011'):
        rows = [row for row in candidate_events if row['dataset'] == 'kubernetes_audit' and audit_fields(row)['uri'] == '/api/v1/namespaces' and audit_fields(row)['user'] == 'system:admin' and audit_fields(row)['code'] == 200]
        return closest(rows, link['delegated_at'])
    return None

def proof_summary(row):
    if row['dataset'] == 'container_log':
        return 'HTTP GET / returned 200 to curl on the vulnerable service'
    if row['dataset'] == 'tetragon':
        process = exec_process(row)
        binary = process.get('binary', row.get('process', ''))
        arguments = process.get('arguments', '')
        uid = process.get('uid', '?')
        pod = process.get('pod', {}).get('name', row.get('pod_name', ''))
        command = f"{binary} {arguments}".strip()
        return f"exec {command} as uid={uid} in pod {pod}"
    if row['dataset'] == 'kubernetes_audit':
        fields = audit_fields(row)
        target = fields['resource'] + (f" in {fields['namespace']}" if fields['namespace'] else '')
        return f"{fields['verb']} {target} as {fields['user']} -> HTTP {fields['code']}"
    return row['dataset']

victim_evidence_by_ability = {}
attack_timeline_rows = []
for index, link in enumerate(links, 1):
    label, marker = ABILITY_META.get(link['ability_id'], (link['ability_id'], ''))
    proof = victim_proof(link)
    assert proof, f"no content-matched victim event for {link['ability_id']}"
    victim_evidence_by_ability[link['ability_id']] = proof
    marker_matches = sum(1 for row in candidate_events if marker and marker in (row.get('raw_data') or ''))
    result = 'success' if str(link['status']) == '0' else f"status {link['status']}"
    attack_timeline_rows.append({
        'step': index,
        'attack_time': link['delegated_at'],
        'attack_action': label,
        'result': result,
        'victim_event_time': proof['event_time'],
        'exact_victim_log': proof_summary(proof),
        'source_event_id': proof['event_id'],
        'marker_matches': marker_matches,
    })
display(Markdown('### Compact chronological attack timeline'))
markdown_table(attack_timeline_rows, ['step', 'attack_time', 'attack_action', 'result', 'victim_event_time', 'exact_victim_log', 'source_event_id', 'marker_matches'])
display(Markdown('`marker_matches` = logs containing that step’s unique marker; `0` does not mean the step failed.'))


In [ ]:
def evidence_graph_svg(operation_id, links, victim_evidence_by_ability):
    width, row_height, top = 1260, 138, 86
    height = top + row_height * len(links) + 30
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
             '<style>text{font-family:Arial,sans-serif;font-size:12px}.title{font-size:15px;font-weight:bold}.small{font-size:10px}.ability{fill:#e8f1ff;stroke:#285f9e;stroke-width:2}.evidence{fill:#eef8ed;stroke:#287a35;stroke-width:2}.operation{fill:#fff2cc;stroke:#9c7611;stroke-width:2}.explicit{stroke:#285f9e;stroke-width:2}.confirmed{stroke:#287a35;stroke-width:2}.proof{fill:#145c25;font-weight:bold}</style>',
             f'<rect class="operation" x="20" y="18" width="1220" height="48" rx="8"/>',
             f'<text class="title" x="36" y="39">Caldera operation {html.escape(operation_id)}</text>',
             '<text x="36" y="57">Each right-hand box is one content-matched victim record, with its original timestamp and event ID—not a window count.</text>']
    for i, link in enumerate(links):
        y = top + i * row_height
        label, marker = ABILITY_META.get(link['ability_id'], (link['ability_id'], ''))
        proof = victim_evidence_by_ability[link['ability_id']]
        summary = proof_summary(proof)
        process = exec_process(proof)
        pod = process.get('pod', {}).get('name', proof.get('pod_name', '')) or 'victim workload'
        parts += [f'<line class="explicit" x1="190" y1="66" x2="190" y2="{y}"/>',
                  f'<rect class="ability" x="20" y="{y}" width="410" height="108" rx="8"/>',
                  f'<text class="title" x="36" y="{y+22}">{i+1}. {html.escape(label)}</text>',
                  f'<text x="36" y="{y+43}">Caldera status {html.escape(str(link["status"]))} · {link["duration_ms"]} ms</text>',
                  f'<text class="small" x="36" y="{y+62}">attack {html.escape(str(link["delegated_at"]))}</text>',
                  f'<text class="small" x="36" y="{y+80}">link {html.escape(link["link_id"])}</text>',
                  f'<line class="confirmed" x1="430" y1="{y+54}" x2="465" y2="{y+54}"/>',
                  f'<rect class="evidence" x="465" y="{y}" width="775" height="108" rx="8"/>',
                  f'<text class="title" x="481" y="{y+21}">Exact victim-side event</text>',
                  f'<text x="481" y="{y+43}">{html.escape(str(proof["event_time"]))} · {html.escape(proof["dataset"])}</text>',
                  f'<text class="small" x="481" y="{y+61}">event {html.escape(proof["event_id"])}</text>',
                  f'<text class="small" x="481" y="{y+78}">k3s-vuln · {html.escape(pod)}</text>',
                  f'<text class="proof" x="481" y="{y+97}">{html.escape(summary)}</text>']
    parts.append('</svg>')
    return ''.join(parts)
display(SVG(evidence_graph_svg(operation_id, links, victim_evidence_by_ability)))


### How to read this graph

The left box is Caldera execution ground truth. The right box is one victim-side record selected by action-specific fields, with its real timestamp and source event ID for lookup. `marker_matches` only counts logs containing the step marker; it is supporting correlation, not the proof row.


## 5. Sealed network artifact and Zeek observations

In [ ]:
artifacts = query(f"SELECT run_id, artifact_id, capture_interface, pcap_sha256, size_bytes, packet_count, zeek_version, zeek_event_count, sealed FROM ir.pcap_artifacts WHERE run_id = '{RUN_ID}'")
assert len(artifacts) == 1 and artifacts[0]['sealed'], artifacts
markdown_table(artifacts, ['artifact_id', 'size_bytes', 'packet_count', 'zeek_event_count', 'sealed'])
connections = query(f"SELECT src_ip, dst_ip, dst_port, protocol, count() AS connections, min(event_time) AS first_seen, max(event_time) AS last_seen FROM ir.network_connections WHERE run_id = '{RUN_ID}' GROUP BY src_ip, dst_ip, dst_port, protocol ORDER BY first_seen LIMIT 30")
markdown_table(connections, ['src_ip', 'dst_ip', 'dst_port', 'protocol', 'connections', 'first_seen', 'last_seen'])

## 6. Runtime lineage and transparent correlation

In [ ]:
lineage = query(f"SELECT namespace, pod_name, process, count() AS events, min(event_time) AS first_seen, max(event_time) AS last_seen FROM ir.process_lineage WHERE run_id = '{RUN_ID}' GROUP BY namespace, pod_name, process ORDER BY first_seen")
markdown_table(lineage, ['namespace', 'pod_name', 'process', 'events', 'first_seen', 'last_seen'])
correlations = query(f"SELECT kubernetes_event_id, tetragon_event_id, kubernetes_time, tetragon_time, delta_ms, namespace, pod_name, process, exec_id, parent_exec_id, correlation_method FROM ir.kubernetes_tetragon_correlation WHERE run_id = '{RUN_ID}' ORDER BY delta_ms LIMIT 100")
if not correlations:
    display(Markdown('_No Kubernetes/Tetragon correlation matched the explicit view rule; source evidence remains available independently._'))
else:
    markdown_table(correlations[:30], ['kubernetes_event_id', 'tetragon_event_id', 'delta_ms', 'namespace', 'pod_name', 'process', 'exec_id', 'parent_exec_id', 'correlation_method'])

## 7. Detections with cited source events

In [ ]:
detections = query(f"SELECT time, rule_id, rule_name, severity, matched_source, matched_host, matched_event_ids FROM ir.detections WHERE experiment_run = '{RUN_ID}' ORDER BY time")
for row in detections:
    row['matched_events'] = len(row.get('matched_event_ids', []))
markdown_table(detections, ['time', 'rule_name', 'severity', 'matched_source', 'matched_host', 'matched_events'])

## 8. Integrity summary

In [ ]:
integrity = query(f"SELECT count() AS timeline_rows, uniqExact(event_id) AS unique_event_ids, countIf(event_id = '') AS missing_event_ids, countIf(membership_method = 'explicit_run_id') AS explicit_rows, countIf(membership_method = 'range_time_window') AS window_rows FROM ir.timeline WHERE run_id = '{RUN_ID}'")[0]
assert integrity['timeline_rows'] > 0
assert integrity['missing_event_ids'] == 0
summary = {'run': runs[0], 'coverage': coverage, 'integrity': integrity, 'pcap_artifact': artifacts[0], 'correlations': len(correlations)}
display(Markdown('```json\n' + json.dumps(summary, indent=2, sort_keys=True) + '\n```'))

## Interpretation boundary

Rows sharing a run ID belong to the same authorized experiment, but that alone does not mean one row caused another. `explicit_run_id` means the producer wrote the run identity into the record. `range_time_window` means VibeRange associated a continuously collected row because it came from the same range between the measured run start and end.

In plain language, Caldera supplies the attack plan and receipt: operation ID, ordered ability ID, unique link ID, delegated/finished times, agent, status, command, and bounded result. Host and network sensors supply independent observations. The ability-to-evidence graph makes the join rule visible and preserves sample event IDs. Solid identifier joins are exact; marker matches are strong supporting observations; time-window matches are candidates for analyst review, not proof of causality.